In [4]:
# ==================== CELL 1: CORRECT SETUP ====================
!pip install indic-nlp-library --quiet

import torch
from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
from indicnlp.tokenize.sentence_tokenize import sentence_split

print("=== Environment Check ===")
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPUs:", torch.cuda.device_count())
!nvidia-smi

# Create normalizer (this is the correct way)
factory = IndicNormalizerFactory()
normalizer_bn = factory.get_normalizer("bn")

print("\n✅ indic-nlp-library loaded successfully!")

=== Environment Check ===
PyTorch version: 2.10.0+cu128
CUDA available: True
GPUs: 2
Sat Aug 29 04:13:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |            

In [5]:
# ==================== CELL 2: CORRECT SMART CHUNKING ====================
def normalize_text(text: str) -> str:
    """Correct way to normalize Bengali text"""
    return normalizer_bn.normalize(text)

def smart_chunk(text: str, lang: str = 'bn') -> list:
    """
    Expert chunking that respects Bengali danda (।)
    """
    normalized = normalize_text(text)
    sentences = sentence_split(normalized, lang=lang)
    return [s.strip() for s in sentences if s.strip()]

# ==================== TEST ====================
print("=== Test 1: Mixed + Danda ===")
test1 = "আমি যাবো। কিন্তু I'm not sure about tomorrow। তুমি কেমন আছো?"
print(smart_chunk(test1))

print("\n=== Test 2: Romanized ===")
test2 = "ami bhalo achi kintu meeting ta miss korte chai na"
print(smart_chunk(test2))

print("\n=== Test 3: With English period ===")
test3 = "আজকের মিটিং ভালো হয়েছে। But I have some doubts."
print(smart_chunk(test3))

=== Test 1: Mixed + Danda ===
['আমি যাবো।', "কিন্তু I'm not sure about tomorrow।", 'তুমি কেমন আছো?']

=== Test 2: Romanized ===
['ami bhalo achi kintu meeting ta miss korte chai na']

=== Test 3: With English period ===
['আজকের মিটিং ভালো হয়েছে।', 'But I have some doubts.']


In [6]:
# ==================== CELL 3: STABLE LANGUAGE DETECTION ====================
!pip install langdetect --quiet

from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0

def detect_language(text: str):
    """
    Simple but reliable language detection.
    Returns 'bn' for Bengali, 'en' for English, 'mixed' if unsure.
    """
    try:
        lang = detect(text)
        # Simple heuristic for romanized Bengali
        romanized_indicators = ['ami', 'tumi', 'kintu', 'kemon', 'bhalo', 'ache', 'jabo', 'korbo']
        text_lower = text.lower()
        
        if lang == 'bn':
            return {"lang": "bn", "script": "Bengali", "note": "Bengali script"}
        elif lang == 'en' and any(word in text_lower for word in romanized_indicators):
            return {"lang": "bn", "script": "Romanized", "note": "Likely romanized Bengali"}
        elif lang == 'en':
            return {"lang": "en", "script": "Latin", "note": "English"}
        else:
            return {"lang": lang, "script": "Other", "note": "Other language"}
    except:
        return {"lang": "unknown", "script": "Unknown", "note": "Detection failed"}

# ==================== TESTS ====================
print("=== Language Detection Tests ===")
print("Bengali script :", detect_language("আমি যাবো"))
print("Romanized      :", detect_language("ami bhalo achi kintu meeting ta miss korte chai na"))
print("English        :", detect_language("I am going to the meeting tomorrow"))
print("Mixed          :", detect_language("ami jabo but I'm not sure about the plan"))
print("Pure Bengali   :", detect_language("আজকের আবহাওয়া খুব ভালো"))

=== Language Detection Tests ===
Bengali script : {'lang': 'bn', 'script': 'Bengali', 'note': 'Bengali script'}
Romanized      : {'lang': 'et', 'script': 'Other', 'note': 'Other language'}
English        : {'lang': 'en', 'script': 'Latin', 'note': 'English'}
Mixed          : {'lang': 'bn', 'script': 'Romanized', 'note': 'Likely romanized Bengali'}
Pure Bengali   : {'lang': 'bn', 'script': 'Bengali', 'note': 'Bengali script'}


In [7]:
# ==================== IMPROVED LANGUAGE DETECTION ====================
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0

def detect_language(text: str):
    text_lower = text.lower()
    
    # Strong romanized Bengali indicators
    romanized_words = ['ami', 'tumi', 'kintu', 'kemon', 'bhalo', 'ache', 'jabo', 'korbo', 
                       'miss', 'korte', 'chai', 'ta', 'na', 'ki']
    
    # Count how many romanized indicators are present
    romanized_score = sum(1 for word in romanized_words if word in text_lower.split())
    
    # If strong signals of romanized Bengali → treat as bn
    if romanized_score >= 2:
        return {"lang": "bn", "script": "Romanized", "note": "Likely romanized Bengali (Banglish)"}
    
    try:
        lang = detect(text)
        if lang == 'bn':
            return {"lang": "bn", "script": "Bengali", "note": "Bengali script"}
        elif lang == 'en':
            return {"lang": "en", "script": "Latin", "note": "English"}
        else:
            return {"lang": lang, "script": "Other", "note": f"Detected as {lang}"}
    except:
        return {"lang": "unknown", "script": "Unknown", "note": "Detection failed"}

# ==================== TESTS ====================
print("=== Language Detection Tests ===")
print("Bengali script :", detect_language("আমি যাবো"))
print("Romanized      :", detect_language("ami bhalo achi kintu meeting ta miss korte chai na"))
print("English        :", detect_language("I am going to the meeting tomorrow"))
print("Mixed          :", detect_language("ami jabo but I'm not sure about the plan"))
print("Pure Bengali   :", detect_language("আজকের আবহাওয়া খুব ভালো"))

=== Language Detection Tests ===
Bengali script : {'lang': 'bn', 'script': 'Bengali', 'note': 'Bengali script'}
Romanized      : {'lang': 'bn', 'script': 'Romanized', 'note': 'Likely romanized Bengali (Banglish)'}
English        : {'lang': 'en', 'script': 'Latin', 'note': 'English'}
Mixed          : {'lang': 'bn', 'script': 'Romanized', 'note': 'Likely romanized Bengali (Banglish)'}
Pure Bengali   : {'lang': 'bn', 'script': 'Bengali', 'note': 'Bengali script'}


In [8]:
# ==================== COMPLETE MODULE: Task 1 ====================
from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
from indicnlp.tokenize.sentence_tokenize import sentence_split
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 0
factory = IndicNormalizerFactory()
normalizer_bn = factory.get_normalizer("bn")

def normalize_text(text: str) -> str:
    return normalizer_bn.normalize(text)

def smart_chunk(text: str, lang: str = 'bn') -> list:
    normalized = normalize_text(text)
    sentences = sentence_split(normalized, lang=lang)
    return [s.strip() for s in sentences if s.strip()]

def detect_language(text: str):
    text_lower = text.lower()
    romanized_words = ['ami', 'tumi', 'kintu', 'kemon', 'bhalo', 'ache', 'jabo', 
                       'korbo', 'miss', 'korte', 'chai', 'ta', 'na', 'ki']
    
    romanized_score = sum(1 for word in romanized_words if word in text_lower.split())
    
    if romanized_score >= 2:
        return {"lang": "bn", "script": "Romanized", "note": "Likely romanized Bengali (Banglish)"}
    
    try:
        lang = detect(text)
        if lang == 'bn':
            return {"lang": "bn", "script": "Bengali", "note": "Bengali script"}
        elif lang == 'en':
            return {"lang": "en", "script": "Latin", "note": "English"}
        else:
            return {"lang": lang, "script": "Other", "note": f"Detected as {lang}"}
    except:
        return {"lang": "unknown", "script": "Unknown", "note": "Detection failed"}

def analyze_text(text: str):
    """
    Complete Task 1 Module:
    - Splits text into sentences (respecting Bengali danda)
    - Detects language for each chunk (including romanized)
    """
    chunks = smart_chunk(text)
    results = []
    for chunk in chunks:
        lang_info = detect_language(chunk)
        results.append({
            "chunk": chunk,
            "language": lang_info
        })
    return results

# ==================== TEST THE FULL MODULE ====================
test_text = "আমি যাবো। কিন্তু ami bhalo achi na। তুমি কেমন আছো?"

print("=== Full Analysis ===")
result = analyze_text(test_text)
for i, item in enumerate(result):
    print(f"\nChunk {i+1}: {item['chunk']}")
    print(f"Language: {item['language']}")

=== Full Analysis ===

Chunk 1: আমি যাবো।
Language: {'lang': 'bn', 'script': 'Bengali', 'note': 'Bengali script'}

Chunk 2: কিন্তু ami bhalo achi na।
Language: {'lang': 'bn', 'script': 'Romanized', 'note': 'Likely romanized Bengali (Banglish)'}

Chunk 3: তুমি কেমন আছো?
Language: {'lang': 'bn', 'script': 'Bengali', 'note': 'Bengali script'}


In [9]:
# ==================== FINAL FIX: Load NLLB-200 ====================
!pip install transformers==4.41.2 sentencepiece --quiet

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

print("Loading NLLB-200 model... Please wait (1-2 minutes)")

model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print(f"✅ NLLB-200 loaded successfully on {device.upper()}!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 101.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 81.0 MB/s eta 0:00:00:00:01
Loading NLLB-200 model... Please wait (1-2 minutes)


tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

✅ NLLB-200 loaded successfully on CUDA!


In [10]:
# ==================== STEP 2: Simple Translation Function ====================
def translate_text(text, src_lang="ben_Beng", tgt_lang="eng_Latn"):
    """
    Translate text from one language to another using NLLB
    src_lang = source language (ben_Beng = Bengali)
    tgt_lang = target language (eng_Latn = English)
    """
    tokenizer.src_lang = src_lang
    
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)
    
    with torch.no_grad():
        translated_tokens = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_lang),
            max_length=256
        )
    
    translated_text = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]
    return translated_text


# ==================== TEST ====================
print("Testing translation...")

# Test 1: Simple Bengali
bengali_text = "আমি যাবো।"
english = translate_text(bengali_text)
print(f"Bengali: {bengali_text}")
print(f"English: {english}")

# Test 2: Another sentence
bengali_text2 = "তুমি কেমন আছো?"
english2 = translate_text(bengali_text2)
print(f"\nBengali: {bengali_text2}")
print(f"English: {english2}")

Testing translation...
Bengali: আমি যাবো।
English: I'm going to go.

Bengali: তুমি কেমন আছো?
English: How are you?


In [11]:
# ==================== STEP 3: Full Basic Pipeline ====================
def translate_full_text(text):
    """
    Takes any text (mixed Bengali + English + Romanized)
    and returns translated English with language info
    """
    # Step 1: Analyze (chunk + detect language)
    analyzed = analyze_text(text)
    
    results = []
    
    for item in analyzed:
        chunk = item["chunk"]
        lang_info = item["language"]
        
        # Step 2: Translate based on language
        if lang_info["lang"] == "bn":
            translated = translate_text(chunk, src_lang="ben_Beng")
        else:
            translated = translate_text(chunk, src_lang="eng_Latn")
        
        results.append({
            "original": chunk,
            "language": lang_info["note"],
            "translation": translated
        })
    
    return results


# ==================== TEST ====================
test_text = "আমি যাবো। কিন্তু ami bhalo achi na। তুমি কেমন আছো?"

print("=== Input Text ===")
print(test_text)

print("\n=== Translation Result ===")
output = translate_full_text(test_text)

for item in output:
    print(f"\nOriginal   : {item['original']}")
    print(f"Language   : {item['language']}")
    print(f"Translation: {item['translation']}")

=== Input Text ===
আমি যাবো। কিন্তু ami bhalo achi na। তুমি কেমন আছো?

=== Translation Result ===

Original   : আমি যাবো।
Language   : Bengali script
Translation: I'm going to go.

Original   : কিন্তু ami bhalo achi na।
Language   : Likely romanized Bengali (Banglish)
Translation: But ami bhalo achi na.

Original   : তুমি কেমন আছো?
Language   : Bengali script
Translation: How are you?


In [12]:
# ==================== SIMPLE TRANSLITERATION ====================
def transliterate_romanized_to_bengali(text):
    """
    Convert romanized Bengali (Banglish) to Bengali script
    This is a simple version. We can improve it later.
    """
    # Common romanized to Bengali mappings
    mapping = {
        "ami": "আমি",
        "tumi": "তুমি",
        "kintu": "কিন্তু",
        "kemon": "কেমন",
        "bhalo": "ভালো",
        "achi": "আছি",
        "ache": "আছে",
        "jabo": "যাবো",
        "korbo": "করবো",
        "miss": "মিস",
        "korte": "করতে",
        "chai": "চাই",
        "na": "না",
        "ki": "কি",
        "ta": "টা",
        "e": "এ",
        "o": "ও",
        "ar": "আর",
        "kore": "করে",
        "hoyeche": "হয়েছে",
        "hobe": "হবে",
    }
    
    words = text.split()
    transliterated_words = []
    
    for word in words:
        word_lower = word.lower().strip("।.!?,")
        
        if word_lower in mapping:
            transliterated_words.append(mapping[word_lower])
        else:
            # Keep the word as it is if not in mapping
            transliterated_words.append(word)
    
    return " ".join(transliterated_words)


# ==================== TEST ====================
test_romanized = "ami bhalo achi kintu meeting ta miss korte chai na"
result = transliterate_romanized_to_bengali(test_romanized)

print("Original Romanized :", test_romanized)
print("Transliterated      :", result)

Original Romanized : ami bhalo achi kintu meeting ta miss korte chai na
Transliterated      : আমি ভালো আছি কিন্তু meeting টা মিস করতে চাই না


In [13]:
# ==================== FINAL PIPELINE WITH TRANSLITERATION ====================
def translate_full_text_final(text):
    """
    Best version so far:
    - Uses chunking + language detection
    - Adds transliteration for romanized Bengali
    - Then translates using NLLB
    """
    analyzed = analyze_text(text)
    results = []
    
    for item in analyzed:
        chunk = item["chunk"]
        lang_info = item["language"]
        
        # If the chunk is romanized Bengali → transliterate first
        if lang_info["lang"] == "bn" and lang_info["script"] == "Romanized":
            chunk_for_translation = transliterate_romanized_to_bengali(chunk)
            translated = translate_text(chunk_for_translation, src_lang="ben_Beng")
        elif lang_info["lang"] == "bn":
            # Normal Bengali script
            translated = translate_text(chunk, src_lang="ben_Beng")
        else:
            # English
            translated = translate_text(chunk, src_lang="eng_Latn")
        
        results.append({
            "original": chunk,
            "language": lang_info["note"],
            "transliterated": chunk_for_translation if lang_info["script"] == "Romanized" else chunk,
            "translation": translated
        })
    
    return results


# ==================== TEST ====================
test_text = "আমি যাবো। কিন্তু ami bhalo achi na। তুমি কেমন আছো?"

print("=== Input Text ===")
print(test_text)

print("\n=== Final Translation Result ===")
output = translate_full_text_final(test_text)

for item in output:
    print(f"\nOriginal     : {item['original']}")
    print(f"Language     : {item['language']}")
    if item.get('transliterated') != item['original']:
        print(f"Transliterated: {item['transliterated']}")
    print(f"Translation  : {item['translation']}")

=== Input Text ===
আমি যাবো। কিন্তু ami bhalo achi na। তুমি কেমন আছো?

=== Final Translation Result ===

Original     : আমি যাবো।
Language     : Bengali script
Translation  : I'm going to go.

Original     : কিন্তু ami bhalo achi na।
Language     : Likely romanized Bengali (Banglish)
Transliterated: কিন্তু আমি ভালো আছি না
Translation  : But I'm not well.

Original     : তুমি কেমন আছো?
Language     : Bengali script
Translation  : How are you?


In [14]:
# ==================== BASIC EMOTION + SARCASM ANALYZER ====================
def analyze_emotion_sarcasm(chunk):
    """
    Basic version to detect emotion and sarcasm
    """
    text_lower = chunk.lower()
    
    result = {
        "emotion": "neutral",
        "sarcasm": False,
        "notes": []
    }
    
    # Simple emotion keywords
    if any(word in text_lower for word in ["খুশি", "আনন্দ", "ভালো", "happy", "joy"]):
        result["emotion"] = "joy/happiness"
    elif any(word in text_lower for word in ["দুঃখ", "কষ্ট", "sad", "unhappy", "কষ্ট"]):
        result["emotion"] = "sadness"
    elif any(word in text_lower for word in ["রাগ", "অসন্তুষ্ট", "angry", "রাগান্বিত"]):
        result["emotion"] = "anger"
    elif any(word in text_lower for word in ["ভয়", "আতঙ্ক", "fear", "scared"]):
        result["emotion"] = "fear"
    
    # Simple sarcasm detection (common patterns)
    sarcasm_markers = ["বাহ", "আহা", "দারুণ", "খুব ভালো", "অসাধারণ"]
    if any(marker in text_lower for marker in sarcasm_markers):
        result["sarcasm"] = True
        result["notes"].append("Possible sarcasm detected (common marker found)")
    
    # Add cultural/slang notes (example)
    if "ফালতু" in text_lower or "ঝামেলা" in text_lower:
        result["notes"].append("Contains common Bengali slang")
    
    return result


# ==================== TEST ====================
test_chunks = [
    "আমি খুব খুশি!",
    "বাহ! তোমার কাজটা খুব ভালো হয়েছে।",
    "এই কাজটা ফালতু হয়ে গেছে।"
]

for chunk in test_chunks:
    analysis = analyze_emotion_sarcasm(chunk)
    print(f"Text: {chunk}")
    print(f"Emotion: {analysis['emotion']}, Sarcasm: {analysis['sarcasm']}")
    print(f"Notes: {analysis['notes']}\n")

Text: আমি খুব খুশি!
Emotion: joy/happiness, Sarcasm: False
Notes: []

Text: বাহ! তোমার কাজটা খুব ভালো হয়েছে।
Emotion: joy/happiness, Sarcasm: True
Notes: ['Possible sarcasm detected (common marker found)']

Text: এই কাজটা ফালতু হয়ে গেছে।
Emotion: neutral, Sarcasm: False
Notes: ['Contains common Bengali slang']



In [15]:
# ==================== COMPLETE PIPELINE ====================
def full_pipeline(text):
    """
    Complete system:
    - Chunking + Language Detection
    - Transliteration (for romanized)
    - Translation (NLLB)
    - Emotion + Sarcasm Analysis
    """
    analyzed = analyze_text(text)
    final_results = []
    
    for item in analyzed:
        chunk = item["chunk"]
        lang_info = item["language"]
        
        # Transliteration if needed
        if lang_info["lang"] == "bn" and lang_info["script"] == "Romanized":
            processed_chunk = transliterate_romanized_to_bengali(chunk)
            translated = translate_text(processed_chunk, src_lang="ben_Beng")
        elif lang_info["lang"] == "bn":
            translated = translate_text(chunk, src_lang="ben_Beng")
        else:
            translated = translate_text(chunk, src_lang="eng_Latn")
        
        # Emotion + Sarcasm Analysis
        emotion_analysis = analyze_emotion_sarcasm(chunk)
        
        final_results.append({
            "original": chunk,
            "language": lang_info["note"],
            "translation": translated,
            "emotion": emotion_analysis["emotion"],
            "sarcasm": emotion_analysis["sarcasm"],
            "notes": emotion_analysis["notes"]
        })
    
    return final_results


# ==================== TEST ====================
test_text = "আমি যাবো। কিন্তু ami bhalo achi na। বাহ! তোমার কাজটা খুব ভালো হয়েছে।"

print("=== Input ===")
print(test_text)

print("\n=== Full Analysis & Translation ===")
result = full_pipeline(test_text)

for i, item in enumerate(result):
    print(f"\n--- Chunk {i+1} ---")
    print(f"Original   : {item['original']}")
    print(f"Language   : {item['language']}")
    print(f"Translation: {item['translation']}")
    print(f"Emotion    : {item['emotion']}")
    print(f"Sarcasm    : {item['sarcasm']}")
    if item['notes']:
        print(f"Notes      : {item['notes']}")

=== Input ===
আমি যাবো। কিন্তু ami bhalo achi na। বাহ! তোমার কাজটা খুব ভালো হয়েছে।

=== Full Analysis & Translation ===

--- Chunk 1 ---
Original   : আমি যাবো।
Language   : Bengali script
Translation: I'm going to go.
Emotion    : neutral
Sarcasm    : False

--- Chunk 2 ---
Original   : কিন্তু ami bhalo achi na।
Language   : Likely romanized Bengali (Banglish)
Translation: But I'm not well.
Emotion    : neutral
Sarcasm    : False

--- Chunk 3 ---
Original   : বাহ!
Language   : Bengali script
Translation: Oh, my God!
Emotion    : neutral
Sarcasm    : True
Notes      : ['Possible sarcasm detected (common marker found)']

--- Chunk 4 ---
Original   : তোমার কাজটা খুব ভালো হয়েছে।
Language   : Bengali script
Translation: You did a great job.
Emotion    : joy/happiness
Sarcasm    : True
Notes      : ['Possible sarcasm detected (common marker found)']


In [16]:
# ==================== IMPROVED SARCASM ANALYZER ====================
import re

def analyze_emotion_sarcasm_v3(chunk, translated_text=""):
    """
    Stronger sarcasm detection with deadpan support
    """
    text_lower = chunk.lower()
    translated_lower = translated_text.lower() if translated_text else ""
    
    result = {
        "emotion": "neutral",
        "sarcasm": False,
        "notes": []
    }
    
    # === Expanded Emotion Keywords ===
    if any(w in text_lower for w in ["খুশি", "আনন্দ", "ভালো", "happy", "joy"]):
        result["emotion"] = "joy/happiness"
    elif any(w in text_lower for w in ["দুঃখ", "কষ্ট", "sad", "unhappy"]):
        result["emotion"] = "sadness"
    elif any(w in text_lower for w in ["রাগ", "অসন্তুষ্ট", "angry"]):
        result["emotion"] = "anger"
    elif any(w in text_lower for w in ["ভয়", "আতঙ্ক"]):
        result["emotion"] = "fear"
    
    # === Improved Sarcasm Detection ===
    sarcasm_markers = ["বাহ", "আহা", "দারুণ", "খুব ভালো", "অসাধারণ", "সুন্দর"]
    if any(m in text_lower for m in sarcasm_markers):
        result["sarcasm"] = True
        result["notes"].append("Sarcasm marker detected")
    
    # Deadpan / Inversion Check (positive Bengali vs negative English)
    positive_bengali = any(w in text_lower for w in ["ভালো", "দারুণ", "সুন্দর", "অসাধারণ"])
    negative_english = any(w in translated_lower for w in ["bad", "terrible", "waste", "not well", "poor", "ruined", "disaster"])
    
    if positive_bengali and negative_english:
        result["sarcasm"] = True
        result["notes"].append("Deadpan sarcasm detected (positive source, negative translation)")
    
    # Contrast patterns (e.g., positive + negative in same sentence)
    if re.search(r'ভালো.*না|দারুণ.*না', text_lower) or re.search(r'good.*not|great.*but', translated_lower):
        result["sarcasm"] = True
        result["notes"].append("Contrast pattern detected (likely sarcasm)")
    
    # === Slang / Cultural Notes ===
    slang_patterns = [r"ফালতু", r"ঝামেলা", r"বাজে", r"অপদার্থ"]
    for pattern in slang_patterns:
        if re.search(pattern, chunk):
            result["notes"].append("Contains Bengali slang/cultural expression")
            break
    
    return result

In [3]:
!pip install -q sentencepiece

In [4]:
import re
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

BN_RANGE = re.compile(r"[\u0980-\u09FF]")
LATIN_WORD = re.compile(r"[A-Za-z]+")

ROMAN_BN_LEXICON = {
    "ami", "tumi", "apni", "amra", "tara", "se", "o", "ei", "oi",
    "bhalo", "kharap", "valo", "khub", "onek", "ektu", "na", "nai",
    "achi", "acho", "ache", "achhe", "achhi", "jaabo", "jabo", "jacchi",
    "kori", "korchi", "korechi", "hoyeche", "hoise",
    "kemon", "keno", "kothay", "kobe", "ki", "kichu", "kaj", "kajta",
    "fultu", "faltu", "baje", "sundor", "dhonnobad",
    "bhai", "bon", "maa", "baba", "bondhu", "prem", "bhalobashi",
    "beshi", "kom", "thik", "hobe", "pari", "parbo", "parchi", "kintu",
}

ROMAN_TO_BN = {
    "ami": "আমি", "tumi": "তুমি", "apni": "আপনি", "amra": "আমরা",
    "bhalo": "ভালো", "valo": "ভালো", "kharap": "খারাপ", "khub": "খুব",
    "onek": "অনেক", "na": "না", "nai": "নাই",
    "achi": "আছি", "acho": "আছো", "ache": "আছে", "achhe": "আছে", "achhi": "আছি",
    "jabo": "যাবো", "jaabo": "যাবো", "kemon": "কেমন", "keno": "কেন",
    "kothay": "কোথায়", "ki": "কী", "kaj": "কাজ", "kajta": "কাজটা",
    "faltu": "ফালতু", "fultu": "ফালতু", "baje": "বাজে", "sundor": "সুন্দর",
    "dhonnobad": "ধন্যবাদ", "bondhu": "বন্ধু", "thik": "ঠিক",
    "hobe": "হবে", "pari": "পারি", "parbo": "পারবো", "kintu": "কিন্তু",
}

POS_EN = {
    "good", "great", "excellent", "amazing", "wonderful", "beautiful", "love",
    "happy", "glad", "nice", "well", "perfect", "best", "awesome", "bravo",
    "fine", "ok", "okay",
}
NEG_EN = {
    "bad", "terrible", "awful", "hate", "sad", "angry", "useless", "worst",
    "not", "never", "no", "poor", "fail", "failed", "rubbish", "trash",
    "pathetic", "horrible", "disappointed", "ugly",
}
POS_BN = {"ভালো", "সুন্দর", "দারুণ", "বাহ", "চমৎকার", "অসাধারণ", "ধন্যবাদ"}
NEG_BN = {"খারাপ", "ফালতু", "বাজে", "না", "নাই", "ঘৃণা", "রাগ", "দুঃখ", "ব্যর্থ"}
SARCASM_MARKERS_BN = ("বাহ", "আহা", "দারুণ", "অসাধারণ")
SARCASM_MARKERS_EN = ("wow", "great job", "yeah right", "bravo")

_model = None
_tokenizer = None


def split_chunks(text):
    text = (text or "").strip()
    if not text:
        return []
    parts = re.split(r"(?<=[।!?\n])\s+|(?<=[.!?])\s+", text)
    chunks = [p.strip() for p in parts if p and p.strip()]
    return chunks or [text]


def detect_lang(chunk):
    bn_chars = len(BN_RANGE.findall(chunk))
    latin_words = LATIN_WORD.findall(chunk)
    roman_hits = sum(1 for w in latin_words if w.lower() in ROMAN_BN_LEXICON)
    if bn_chars >= 2 and roman_hits == 0:
        return {"lang": "bn", "script": "Bengali", "note": "Bengali (script)"}
    if roman_hits >= 1 and bn_chars == 0:
        return {"lang": "bn", "script": "Romanized", "note": "Bengali (romanized)"}
    if bn_chars >= 1 and roman_hits >= 1:
        return {"lang": "bn", "script": "Mixed", "note": "Bengali (mixed / code-switch)"}
    if latin_words and roman_hits == 0:
        return {"lang": "en", "script": "Latin", "note": "English"}
    if bn_chars:
        return {"lang": "bn", "script": "Bengali", "note": "Bengali (script)"}
    return {"lang": "en", "script": "Latin", "note": "English / other"}


def analyze_text(text):
    return [{"chunk": c, "language": detect_lang(c)} for c in split_chunks(text)]


def transliterate_romanized_to_bengali(chunk):
    def repl(m):
        return ROMAN_TO_BN.get(m.group(0).lower(), m.group(0))
    return LATIN_WORD.sub(repl, chunk)


def get_model():
    global _model, _tokenizer
    if _model is None:
        name = "facebook/nllb-200-distilled-600M"
        device = "cuda" if torch.cuda.is_available() else "cpu"
        dtype = torch.float16 if device == "cuda" else torch.float32
        print("Loading NLLB-200 (first time can take 1–3 min)...")
        _tokenizer = AutoTokenizer.from_pretrained(name)
        _model = AutoModelForSeq2SeqLM.from_pretrained(name, torch_dtype=dtype)
        _model.to(device)
        _model.eval()
        print("Model ready on", device)
    return _model, _tokenizer


def translate_text(text, src_lang="ben_Beng"):
    text = (text or "").strip()
    if not text:
        return ""
    model, tokenizer = get_model()
    tokenizer.src_lang = src_lang
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    forced_bos = tokenizer.convert_tokens_to_ids("eng_Latn")
    with torch.no_grad():
        out = model.generate(**inputs, forced_bos_token_id=forced_bos, max_length=512)
    return tokenizer.batch_decode(out, skip_special_tokens=True)[0]


def _tokens(s):
    return re.findall(r"[A-Za-z']+|[\u0980-\u09FF]+", (s or "").lower())


def analyze_emotion_sarcasm_v2(original, translated):
    o = original or ""
    t = translated or ""
    ot = _tokens(o)
    tt = _tokens(t)
    pos = sum(1 for w in tt if w in POS_EN) + sum(1 for w in ot if w in POS_BN)
    neg = sum(1 for w in tt if w in NEG_EN) + sum(1 for w in ot if w in NEG_BN)
    if re.search(r"\b(not|never|no|don't|dont|isn't|isnt)\b", t.lower()):
        if pos and not neg:
            neg += 1
            pos = max(0, pos - 1)
    if pos > neg and pos >= 1:
        emotion = "Positive"
    elif neg > pos and neg >= 1:
        emotion = "Negative"
    elif pos == 0 and neg == 0:
        emotion = "Neutral"
    else:
        emotion = "Mixed"
    notes = []
    sarcasm = "No"
    has_praise = any(m in o for m in SARCASM_MARKERS_BN) or any(m in t.lower() for m in SARCASM_MARKERS_EN)
    contrast = ("কিন্তু" in o) or ("but" in t.lower())
    if has_praise and (emotion in {"Negative", "Mixed"} or neg >= 1):
        sarcasm = "Yes"
        notes.append("Praise marker + negative sense (likely sarcastic).")
    elif contrast and pos >= 1 and neg >= 1:
        sarcasm = "Possible"
        notes.append("Contrast (কিন্তু / but) with mixed polarity.")
    elif emotion == "Positive" and any(w in o for w in NEG_BN):
        sarcasm = "Possible"
        notes.append("Positive wording over a negative Bangla cue.")
    return {"emotion": emotion, "sarcasm": sarcasm, "notes": " ".join(notes)}


def full_pipeline_v2(text):
    analyzed = analyze_text(text)
    final_results = []
    for item in analyzed:
        chunk = item["chunk"]
        lang_info = item["language"]
        if lang_info["lang"] == "bn" and lang_info["script"] in ("Romanized", "Mixed"):
            processed_chunk = transliterate_romanized_to_bengali(chunk)
            translated = translate_text(processed_chunk, src_lang="ben_Beng")
        elif lang_info["lang"] == "bn":
            translated = translate_text(chunk, src_lang="ben_Beng")
        else:
            translated = translate_text(chunk, src_lang="eng_Latn")
        emotion_analysis = analyze_emotion_sarcasm_v2(chunk, translated)
        final_results.append({
            "original": chunk,
            "language": lang_info["note"],
            "translation": translated,
            "emotion": emotion_analysis["emotion"],
            "sarcasm": emotion_analysis["sarcasm"],
            "notes": emotion_analysis["notes"],
        })
    return final_results


test_text = "আমি যাবো। কিন্তু ami bhalo achi na। বাহ! তোমার কাজটা খুব ভালো হয়েছে। এই কাজটা ফালতু হয়ে গেছে।"
print("=== Input ===")
print(test_text)
print("\n=== Full Analysis ===")
result = full_pipeline_v2(test_text)
for i, item in enumerate(result):
    print(f"\n--- Chunk {i+1} ---")
    print(f"Original   : {item['original']}")
    print(f"Language   : {item['language']}")
    print(f"Translation: {item['translation']}")
    print(f"Emotion    : {item['emotion']}")
    print(f"Sarcasm    : {item['sarcasm']}")
    if item["notes"]:
        print(f"Notes      : {item['notes']}")

=== Input ===
আমি যাবো। কিন্তু ami bhalo achi na। বাহ! তোমার কাজটা খুব ভালো হয়েছে। এই কাজটা ফালতু হয়ে গেছে।

=== Full Analysis ===
Loading NLLB-200 (first time can take 1–3 min)...


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Model ready on cuda

--- Chunk 1 ---
Original   : আমি যাবো।
Language   : Bengali (script)
Translation: I'm going to go.
Emotion    : Neutral
Sarcasm    : No

--- Chunk 2 ---
Original   : কিন্তু ami bhalo achi na।
Language   : Bengali (mixed / code-switch)
Translation: But I'm not well.
Emotion    : Mixed
Sarcasm    : Possible
Notes      : Contrast (কিন্তু / but) with mixed polarity.

--- Chunk 3 ---
Original   : বাহ!
Language   : Bengali (script)
Translation: Wow!
Emotion    : Positive
Sarcasm    : No

--- Chunk 4 ---
Original   : তোমার কাজটা খুব ভালো হয়েছে।
Language   : Bengali (script)
Translation: You did a very good job.
Emotion    : Positive
Sarcasm    : No

--- Chunk 5 ---
Original   : এই কাজটা ফালতু হয়ে গেছে।
Language   : Bengali (script)
Translation: This is crazy.
Emotion    : Negative
Sarcasm    : No


In [5]:
# ==================== COMPLETE FINAL PIPELINE FOR BAISHALI ====================

# === 1. Setup (already done in your notebook) ===
# Make sure you have the previous functions: smart_chunk, detect_language, transliterate_romanized_to_bengali, translate_text, analyze_emotion_sarcasm_v2

# === 2. Final Full Pipeline ===
def full_pipeline_for_baishali(text):
    """
    Final refined pipeline for Baishali
    Returns clean, structured output with translation + emotion + notes
    """
    analyzed = analyze_text(text)
    final_results = []
    
    for item in analyzed:
        chunk = item["chunk"]
        lang_info = item["language"]
        
        # Transliteration for romanized
        if lang_info["lang"] == "bn" and lang_info["script"] == "Romanized":
            processed_chunk = transliterate_romanized_to_bengali(chunk)
            translated = translate_text(processed_chunk, src_lang="ben_Beng")
        elif lang_info["lang"] == "bn":
            translated = translate_text(chunk, src_lang="ben_Beng")
        else:
            translated = translate_text(chunk, src_lang="eng_Latn")
        
        # Emotion + Sarcasm + Cultural Analysis
        emotion_analysis = analyze_emotion_sarcasm_v2(chunk, translated)
        
        final_results.append({
            "original": chunk,
            "language": lang_info["note"],
            "translation": translated,
            "emotion": emotion_analysis["emotion"],
            "sarcasm": emotion_analysis["sarcasm"],
            "notes": emotion_analysis["notes"]
        })
    
    return final_results


# ==================== USAGE EXAMPLE ====================
test_text = """আমি যাবো। কিন্তু ami bhalo achi na। বাহ! তোমার কাজটা খুব ভালো হয়েছে। এই কাজটা ফালতু হয়ে গেছে।"""

print("=== INPUT ===")
print(test_text)

print("\n=== OUTPUT FOR BAISHALI ===")
result = full_pipeline_for_baishali(test_text)

for i, item in enumerate(result):
    print(f"\n--- Chunk {i+1} ---")
    print(f"Original     : {item['original']}")
    print(f"Translation  : {item['translation']}")
    print(f"Emotion      : {item['emotion']}")
    print(f"Sarcasm      : {item['sarcasm']}")
    if item['notes']:
        print(f"Notes        : {item['notes']}")

=== INPUT ===
আমি যাবো। কিন্তু ami bhalo achi na। বাহ! তোমার কাজটা খুব ভালো হয়েছে। এই কাজটা ফালতু হয়ে গেছে।

=== OUTPUT FOR BAISHALI ===

--- Chunk 1 ---
Original     : আমি যাবো।
Translation  : I'm going to go.
Emotion      : Neutral
Sarcasm      : No

--- Chunk 2 ---
Original     : কিন্তু ami bhalo achi na।
Translation  : But I am not a man.
Emotion      : Negative
Sarcasm      : No

--- Chunk 3 ---
Original     : বাহ!
Translation  : Wow!
Emotion      : Positive
Sarcasm      : No

--- Chunk 4 ---
Original     : তোমার কাজটা খুব ভালো হয়েছে।
Translation  : You did a very good job.
Emotion      : Positive
Sarcasm      : No

--- Chunk 5 ---
Original     : এই কাজটা ফালতু হয়ে গেছে।
Translation  : This is crazy.
Emotion      : Negative
Sarcasm      : No


In [6]:
# ==================== FINAL REFINED PIPELINE ====================

def full_pipeline_refined(text):
    """
    Refined production-ready pipeline for Baishali
    """
    analyzed = analyze_text(text)
    final_results = []
    
    for item in analyzed:
        chunk = item["chunk"]
        lang_info = item["language"]
        
        # Transliteration for romanized Bengali
        if lang_info["lang"] == "bn" and lang_info["script"] == "Romanized":
            processed_chunk = transliterate_romanized_to_bengali(chunk)
            translated = translate_text(processed_chunk, src_lang="ben_Beng")
        elif lang_info["lang"] == "bn":
            translated = translate_text(chunk, src_lang="ben_Beng")
        else:
            translated = translate_text(chunk, src_lang="eng_Latn")
        
        # Improved Emotion + Sarcasm + Cultural Analysis
        emotion_analysis = analyze_emotion_sarcasm_v2(chunk, translated)
        
        final_results.append({
            "original": chunk,
            "language": lang_info["note"],
            "translation": translated,
            "emotion": emotion_analysis["emotion"],
            "sarcasm": emotion_analysis["sarcasm"],
            "notes": emotion_analysis["notes"]
        })
    
    return final_results


# ==================== BEAUTIFUL OUTPUT FOR BAISHALI ====================
def display_for_baishali(text):
    result = full_pipeline_refined(text)
    
    print("=" * 60)
    print("TRANSLATION + EMOTION ANALYSIS FOR BAISHALI")
    print("=" * 60)
    
    for i, item in enumerate(result, 1):
        print(f"\n📌 Chunk {i}")
        print(f"Original     : {item['original']}")
        print(f"Translation  : {item['translation']}")
        print(f"Emotion      : {item['emotion']}")
        print(f"Sarcasm      : {'Yes ⚠️' if item['sarcasm'] else 'No'}")
        if item['notes']:
            print(f"Notes        : {', '.join(item['notes'])}")
        print("-" * 50)
    
    print("\n✅ Analysis Complete")


# ==================== TEST ====================
test_text = "আমি যাবো। কিন্তু ami bhalo achi na। বাহ! তোমার কাজটা খুব ভালো হয়েছে। এই কাজটা ফালতু হয়ে গেছে।"

print("=== INPUT ===")
print(test_text)

print("\n")
display_for_baishali(test_text)

=== INPUT ===
আমি যাবো। কিন্তু ami bhalo achi na। বাহ! তোমার কাজটা খুব ভালো হয়েছে। এই কাজটা ফালতু হয়ে গেছে।


TRANSLATION + EMOTION ANALYSIS FOR BAISHALI

📌 Chunk 1
Original     : আমি যাবো।
Translation  : I'm going to go.
Emotion      : Neutral
Sarcasm      : Yes ⚠️
--------------------------------------------------

📌 Chunk 2
Original     : কিন্তু ami bhalo achi na।
Translation  : But I am not a man.
Emotion      : Negative
Sarcasm      : Yes ⚠️
--------------------------------------------------

📌 Chunk 3
Original     : বাহ!
Translation  : Wow!
Emotion      : Positive
Sarcasm      : Yes ⚠️
--------------------------------------------------

📌 Chunk 4
Original     : তোমার কাজটা খুব ভালো হয়েছে।
Translation  : You did a very good job.
Emotion      : Positive
Sarcasm      : Yes ⚠️
--------------------------------------------------

📌 Chunk 5
Original     : এই কাজটা ফালতু হয়ে গেছে।
Translation  : This is crazy.
Emotion      : Negative
Sarcasm      : Yes ⚠️
--------------------------------

In [7]:
# ==================== HARDENED FINAL VERSION ====================

!pip install gradio --quiet

import gradio as gr
import re

# ==================== SAFE WRAPPERS ====================
def safe_detect_language(text):
    """Always returns a dict, never None"""
    try:
        result = detect_language(text)
        if result is None:
            return {"lang": "unknown", "note": "Unknown", "script": "Unknown"}
        return result
    except:
        return {"lang": "unknown", "note": "Unknown", "script": "Unknown"}

def safe_analyze_emotion(text, translated=""):
    try:
        return analyze_emotion_sarcasm_v2(text, translated)
    except:
        return {"emotion": "neutral", "sarcasm": False, "notes": []}

# ==================== GLOBAL ANALYSIS ====================
def analyze_global_context(full_text):
    try:
        return analyze_emotion_sarcasm_v2(full_text, "")
    except:
        return {"emotion": "neutral", "sarcasm": False, "notes": []}

# ==================== MAIN PIPELINE (HARDENED) ====================
def full_pipeline_baishali(text, direction="Auto Detect"):
    if not text or not text.strip():
        return []
    
    global_analysis = analyze_global_context(text)
    
    # Smarter chunking
    chunks = re.split(r'(?<=[।.!?])\s+', text.strip())
    chunks = [c.strip() for c in chunks if c.strip()]
    
    results = []
    
    for chunk in chunks:
        try:
            lang_info = safe_detect_language(chunk)
            
            # Translation direction
            if direction == "Bengali → English" or (direction == "Auto Detect" and lang_info["lang"] == "bn"):
                if lang_info.get("script") == "Romanized":
                    processed = transliterate_romanized_to_bengali(chunk)
                    translated = translate_text(processed, src_lang="ben_Beng")
                else:
                    translated = translate_text(chunk, src_lang="ben_Beng")
            else:
                # English → Bengali
                translated = translate_text(chunk, src_lang="eng_Latn", tgt_lang="ben_Beng")
            
            analysis = safe_analyze_emotion(chunk, translated)
            
            # Sarcasm override
            is_sarcastic = analysis.get("sarcasm", False) or global_analysis.get("sarcasm", False)
            final_emotion = "Sarcastic / Irony" if is_sarcastic else analysis.get("emotion", "neutral")
            
            # Clean notes
            all_notes = list(set(analysis.get("notes", []) + global_analysis.get("notes", [])))
            
            results.append({
                "original": chunk,
                "language": lang_info.get("note", "Unknown"),
                "translation": translated,
                "emotion": final_emotion,
                "sarcasm": is_sarcastic,
                "notes": all_notes
            })
        except Exception as e:
            results.append({
                "original": chunk,
                "language": "Error",
                "translation": f"[Error: {str(e)}]",
                "emotion": "unknown",
                "sarcasm": False,
                "notes": []
            })
    
    return results

# ==================== GRADIO UI ====================
def run_ui(text, direction):
    if not text.strip():
        return "Please enter some text."
    
    result = full_pipeline_baishali(text, direction)
    
    output = ""
    for i, item in enumerate(result, 1):
        output += f"--- Chunk {i} ---\n"
        output += f"Original     : {item['original']}\n"
        output += f"Translation  : {item['translation']}\n"
        output += f"Emotion      : {item['emotion']}\n"
        output += f"Sarcasm      : {'Yes ⚠️' if item['sarcasm'] else 'No'}\n"
        if item['notes']:
            output += f"Notes        : {', '.join(item['notes'])}\n"
        output += "\n"
    
    return output.strip() or "No output."

with gr.Blocks(title="Bengali ↔ English + Analysis (Hardened)") as demo:
    gr.Markdown("# Bengali ↔ English Translator + Emotion/Sarcasm Analysis")
    gr.Markdown("Global context • Smart chunking • Bidirectional • Stable")
    
    input_text = gr.Textbox(label="Enter Text", lines=8)
    direction = gr.Radio(["Bengali → English", "English → Bengali", "Auto Detect"], 
                         value="Auto Detect", label="Direction")
    
    btn = gr.Button("Translate & Analyze", variant="primary")
    output = gr.Textbox(label="Result", lines=22)
    
    btn.click(fn=run_ui, inputs=[input_text, direction], outputs=output)

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://a83a2d0a39b7cf1607.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
